# 🧠 SQL Practice — HR Analytics Employee Attrition

**Dataset:** [HR Analytics Employee Attrition & Performance — Kaggle](https://www.kaggle.com/datasets/mahmoudemadabdallah/hr-analytics-employee-attrition-and-performance)

Este cuaderno te permite practicar SQL desde nivel básico hasta avanzado sobre un dataset real de Recursos Humanos. Usamos **SQLite en memoria** — no necesitas instalar ninguna base de datos.

---

## 📋 Contenido

| Sección | Queries | Temas |
|---|---|---|
| **Setup** | — | Descarga, carga a SQLite, exploración inicial |
| **Nivel básico** | 18 | SELECT, DISTINCT, LIKE, IN, BETWEEN, agregación, CASE WHEN, HAVING, COALESCE, NULLIF, CAST |
| **Nivel intermedio** | 20 | JOINs, subqueries, CTEs, UNION, INTERSECT, EXCEPT, texto, fechas |
| **Nivel avanzado** | 18 | Window functions, LAG/LEAD, ROWS BETWEEN, PERCENT_RANK, CUME_DIST, Pareto, cohortes |

> **Nota:** ROLLUP no está soportado en SQLite — se menciona en el nivel avanzado como referencia para otros motores.

---

## 🗂️ Modelo de datos

```
employee (1) ──── (N) performance
    │
    ├── Education ──────────────► education_level (EducationLevelID)
    │         performance.JobSatisfaction       ──► satisfied_level (SatisfactionID)
    │         performance.EnvironmentSatisfaction──► satisfied_level
    │         performance.RelationshipSatisfaction►  satisfied_level
    │         performance.WorkLifeBalance        ──► satisfied_level
    │         performance.ManagerRating          ──► rating_level (RatingID)
    │         performance.SelfRating             ──► rating_level
```

| Tabla | Filas | Descripción |
|---|---|---|
| `employee` | 1,470 | Datos demográficos, laborales y de rotación |
| `performance` | 6,709 | Evaluaciones periódicas (hasta 10 por empleado) |
| `education_level` | 5 | Catálogo de niveles educativos |
| `rating_level` | 5 | Catálogo: Unacceptable → Above and Beyond |
| `satisfied_level` | 5 | Catálogo: Very Dissatisfied → Very Satisfied |

> **Tip:** Ejecuta las celdas en orden de arriba hacia abajo. El setup debe completarse antes de correr cualquier query.

---
# ⚙️ SECCIÓN 1 — SETUP
---

## 1.1 Configurar la API de Kaggle

Para descargar el dataset necesitas un archivo `kaggle.json` con tus credenciales.

**¿Cómo obtenerlo?**
1. Entra a [kaggle.com/settings](https://www.kaggle.com/settings)
2. Baja hasta la sección **API**
3. Haz clic en **"Crear clave API heredada"**
4. Se descarga el archivo `kaggle.json` — guárdalo en tu computadora

Luego ejecuta la celda de abajo y selecciona ese archivo cuando te lo pida.

In [ ]:
from google.colab import files
import os

files.upload()

os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

print('✓ Kaggle API configurada correctamente')

## 1.2 Descargar el dataset

`kagglehub` descarga automáticamente la versión más reciente del dataset y lo guarda en el sistema de archivos de Colab. El proceso tarda unos segundos.

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    'mahmoudemadabdallah/hr-analytics-employee-attrition-and-performance'
)

print(f'✓ Dataset descargado en: {path}')
print('\nArchivos disponibles:')
for f in os.listdir(path):
    print(f'  - {f}')

## 1.3 Cargar los CSV a SQLite

Creamos una base de datos SQLite **en memoria** (`:memory:`) y cargamos cada CSV como una tabla independiente usando `pandas`.

**¿Por qué SQLite en memoria?**
- No requiere instalación ni configuración de servidor
- Soporta prácticamente toda la sintaxis SQL estándar
- Suficientemente rápido para datasets de este tamaño
- Las queries que escribas aquí son 95% portables a PostgreSQL, SQL Server o BigQuery

In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')

tablas = {
    'employee':        'Employee.csv',
    'performance':     'PerformanceRating.csv',
    'education_level': 'EducationLevel.csv',
    'rating_level':    'RatingLevel.csv',
    'satisfied_level': 'SatisfiedLevel.csv',
}

for nombre_tabla, archivo in tablas.items():
    df = pd.read_csv(f'{path}/{archivo}')
    df.to_sql(nombre_tabla, conn, index=False, if_exists='replace')
    print(f'✓ {nombre_tabla}: {len(df):,} filas — {len(df.columns)} columnas')

print('\n✅ Base de datos lista para consultas')

## 1.4 Función helper `sql()`

En lugar de escribir `pd.read_sql(query, conn)` cada vez, definimos una función corta. Úsala en todas las celdas siguientes para ejecutar cualquier query.

In [ ]:
def sql(query):
    """Ejecuta una query SQL y retorna el resultado como DataFrame."""
    return pd.read_sql(query, conn)

print('✓ Función sql() lista')

## 1.5 Exploración inicial del dataset

In [ ]:
print('=== EMPLOYEE (primeras 3 filas) ===')
display(sql('SELECT * FROM employee LIMIT 3'))

print('\n=== PERFORMANCE (primeras 3 filas) ===')
display(sql('SELECT * FROM performance LIMIT 3'))

In [ ]:
print('=== EDUCATION LEVEL ===')
display(sql('SELECT * FROM education_level'))

print('\n=== RATING LEVEL ===')
display(sql('SELECT * FROM rating_level'))

print('\n=== SATISFIED LEVEL ===')
display(sql('SELECT * FROM satisfied_level'))

In [ ]:
print('Departamentos:', sql('SELECT DISTINCT Department FROM employee')['Department'].tolist())
print('Job Roles:',     sql('SELECT DISTINCT JobRole FROM employee ORDER BY JobRole')['JobRole'].tolist())
print('Viajes:',        sql('SELECT DISTINCT BusinessTravel FROM employee')['BusinessTravel'].tolist())
print('Rango HireDate:', sql('SELECT MIN(HireDate), MAX(HireDate) FROM employee').values[0])

---
# 🟢 SECCIÓN 2 — NIVEL BÁSICO

Funciones practicadas: `SELECT`, `DISTINCT`, `LIKE`, `IN`, `BETWEEN`, `NOT IN`, `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`, `ROUND`, `CASE WHEN`, `GROUP BY`, `ORDER BY`, `HAVING`, `COALESCE`, `NULLIF`, `CAST`, `IS NULL`, `IS NOT NULL`

---

### Bloque 1 — Exploración básica

### Q01 — Selección y exploración con DISTINCT

**Concepto:** `DISTINCT` elimina filas duplicadas del resultado. Es útil para explorar qué valores únicos existen en una columna antes de escribir filtros o segmentaciones.

In [ ]:
sql("""
SELECT DISTINCT
    Department,
    JobRole,
    MaritalStatus
FROM employee
ORDER BY Department, JobRole
""")

### Q02 — Filtrar con LIKE y wildcards

**Concepto:** `LIKE` filtra texto con patrones. El símbolo `%` representa cualquier cantidad de caracteres, y `_` representa exactamente un carácter. Es el equivalente SQL de una búsqueda de texto parcial.

In [ ]:
sql("""
SELECT
    EmployeeID,
    FirstName || ' ' || LastName AS nombre,
    JobRole,
    Department
FROM employee
WHERE JobRole LIKE '%Manager%'
   OR JobRole LIKE '%Engineer%'
ORDER BY JobRole
""")

### Q03 — Filtrar con IN y NOT IN

**Concepto:** `IN` comprueba si un valor pertenece a una lista — es más limpio que encadenar múltiples `OR`. `NOT IN` hace lo opuesto: excluye los valores de la lista. Importante: si la lista puede contener `NULL`, `NOT IN` puede dar resultados inesperados.

In [ ]:
sql("""
SELECT
    EmployeeID,
    FirstName || ' ' || LastName AS nombre,
    Department,
    JobRole,
    Salary
FROM employee
WHERE Department IN ('Sales', 'Human Resources')
  AND JobRole NOT IN ('Manager', 'HR Manager')
ORDER BY Department, Salary DESC
""")

### Q04 — Filtrar con BETWEEN

**Concepto:** `BETWEEN a AND b` filtra valores en un rango inclusivo — equivale a `>= a AND <= b`. Funciona con números, fechas y texto. Es más legible que escribir las dos condiciones por separado.

In [ ]:
sql("""
SELECT
    EmployeeID,
    FirstName || ' ' || LastName AS nombre,
    Age,
    Salary,
    Department,
    HireDate
FROM employee
WHERE Age BETWEEN 30 AND 40
  AND Salary BETWEEN 50000 AND 100000
  AND HireDate BETWEEN '2015-01-01' AND '2019-12-31'
ORDER BY Salary DESC
""")

### Bloque 2 — Funciones de agregación

### Q05 — COUNT, SUM, AVG, MIN, MAX y ROUND

**Concepto:** Las funciones de agregación resumen un conjunto de filas en un único valor. `ROUND(valor, decimales)` controla la precisión. Todas ignoran los valores `NULL` excepto `COUNT(*)` que cuenta todas las filas.

In [ ]:
sql("""
SELECT
    COUNT(*)              AS total_empleados,
    SUM(Salary)           AS masa_salarial,
    ROUND(AVG(Salary), 0) AS salario_promedio,
    MIN(Salary)           AS salario_minimo,
    MAX(Salary)           AS salario_maximo,
    ROUND(AVG(Age), 1)    AS edad_promedio,
    MIN(Age)              AS edad_minima,
    MAX(Age)              AS edad_maxima
FROM employee
""")

### Q06 — Agregación por grupo: salario por departamento

**Concepto:** `GROUP BY` agrupa las filas por los valores únicos de una columna y aplica las funciones de agregación a cada grupo por separado. Cada columna en el `SELECT` debe estar en el `GROUP BY` o dentro de una función de agregación.

In [ ]:
sql("""
SELECT
    Department,
    COUNT(*)              AS total_empleados,
    ROUND(AVG(Salary), 0) AS salario_promedio,
    MIN(Salary)           AS salario_minimo,
    MAX(Salary)           AS salario_maximo,
    SUM(Salary)           AS masa_salarial
FROM employee
GROUP BY Department
ORDER BY salario_promedio DESC
""")

### Q07 — Tasa de rotación con SUM condicional

**Concepto:** `SUM(CASE WHEN condición THEN 1 ELSE 0 END)` cuenta solo las filas que cumplen una condición — es el `COUNTIF` de SQL. Dividirlo entre `COUNT(*)` y multiplicar por 100 da la tasa porcentual.

In [ ]:
sql("""
SELECT
    Department,
    COUNT(*) AS total,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS rotaron,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
    ) AS tasa_rotacion_pct
FROM employee
GROUP BY Department
ORDER BY tasa_rotacion_pct DESC
""")

### Q08 — HAVING: filtrar grupos después de agregar

**Concepto:** `HAVING` filtra sobre el resultado del `GROUP BY`. La diferencia clave con `WHERE`: `WHERE` filtra filas **antes** de agrupar, `HAVING` filtra grupos **después** de agregar. No puedes usar funciones de agregación en el `WHERE`.

In [ ]:
sql("""
SELECT
    JobRole,
    COUNT(*) AS total,
    ROUND(AVG(Salary), 0) AS salario_promedio,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS rotaron,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
    ) AS tasa_rotacion_pct
FROM employee
GROUP BY JobRole
HAVING COUNT(*) > 50
   AND AVG(Salary) > 60000
ORDER BY tasa_rotacion_pct DESC
""")

### Bloque 3 — Segmentación con CASE WHEN

### Q09 — Segmentación por rango de edad y rotación

**Concepto:** `CASE WHEN` con rangos numéricos crea bins sobre una columna continua — equivalente a agrupar valores en intervalos. Puedes usarlo tanto en el `SELECT` como en el `GROUP BY`.

In [ ]:
sql("""
SELECT
    CASE
        WHEN Age < 25              THEN '< 25'
        WHEN Age BETWEEN 25 AND 34 THEN '25-34'
        WHEN Age BETWEEN 35 AND 44 THEN '35-44'
        WHEN Age BETWEEN 45 AND 54 THEN '45-54'
        ELSE '55+'
    END AS rango_edad,
    COUNT(*) AS total,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS rotaron,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
    ) AS tasa_rotacion_pct
FROM employee
GROUP BY rango_edad
ORDER BY rango_edad
""")

### Q10 — Segmentación de salario en bandas

**Concepto:** Mismo patrón de segmentación pero aplicado a una variable de negocio clave — el salario. Crear bandas salariales es un análisis común en RRHH para entender la distribución de compensación.

In [ ]:
sql("""
SELECT
    CASE
        WHEN Salary < 50000                THEN 'Menos de 50K'
        WHEN Salary BETWEEN 50000 AND 79999 THEN '50K - 79K'
        WHEN Salary BETWEEN 80000 AND 109999 THEN '80K - 109K'
        WHEN Salary BETWEEN 110000 AND 139999 THEN '110K - 139K'
        ELSE '140K+'
    END AS banda_salarial,
    COUNT(*) AS empleados,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_total,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS rotaron
FROM employee
GROUP BY banda_salarial
ORDER BY MIN(Salary)
""")

### Q11 — CASE WHEN con múltiples columnas: perfil de riesgo simple

**Concepto:** `CASE WHEN` puede evaluar múltiples columnas en la misma expresión usando `AND` / `OR`. Esto permite crear etiquetas o categorías compuestas que combinan varias condiciones.

In [ ]:
sql("""
SELECT
    EmployeeID,
    FirstName || ' ' || LastName AS nombre,
    OverTime,
    YearsSinceLastPromotion,
    BusinessTravel,
    CASE
        WHEN OverTime = 'Yes'
         AND YearsSinceLastPromotion >= 5
         AND BusinessTravel = 'Frequent Traveller' THEN 'Riesgo Alto'
        WHEN OverTime = 'Yes'
         AND YearsSinceLastPromotion >= 3           THEN 'Riesgo Medio'
        WHEN OverTime = 'No'
         AND YearsSinceLastPromotion < 2            THEN 'Riesgo Bajo'
        ELSE 'Sin clasificar'
    END AS nivel_riesgo,
    Attrition
FROM employee
ORDER BY nivel_riesgo, Attrition
LIMIT 30
""")

### Bloque 4 — Nulos y coerciones

### Q12 — IS NULL e IS NOT NULL

**Concepto:** `NULL` en SQL significa ausencia de valor — no es cero ni cadena vacía. No puedes compararlo con `= NULL` (siempre da falso), debes usar `IS NULL` o `IS NOT NULL`. Detectar nulos es el primer paso de cualquier limpieza de datos.

In [ ]:
sql("""
SELECT
    COUNT(*)                                                    AS total_filas,
    COUNT(EmployeeID)                                           AS con_id,
    SUM(CASE WHEN EmployeeID   IS NULL THEN 1 ELSE 0 END)       AS nulos_id,
    SUM(CASE WHEN Department   IS NULL THEN 1 ELSE 0 END)       AS nulos_department,
    SUM(CASE WHEN Salary       IS NULL THEN 1 ELSE 0 END)       AS nulos_salary,
    SUM(CASE WHEN HireDate     IS NULL THEN 1 ELSE 0 END)       AS nulos_hiredate
FROM employee
""")

### Q13 — COALESCE: reemplazar nulos con un valor por defecto

**Concepto:** `COALESCE(col1, col2, ..., valor_default)` retorna el primer valor no nulo de la lista. Es la forma estándar de manejar nulos en SQL — equivale al `ISNULL()` de SQL Server o `IFNULL()` de MySQL.

In [ ]:
sql("""
SELECT
    EmployeeID,
    FirstName || ' ' || LastName                            AS nombre,
    COALESCE(Department, 'Sin departamento')                AS department,
    COALESCE(Salary, 0)                                     AS salary,
    COALESCE(YearsSinceLastPromotion, 0)                    AS anos_sin_promocion,
    COALESCE(OverTime, 'No registrado')                     AS overtime
FROM employee
ORDER BY EmployeeID
LIMIT 10
""")

### Q14 — NULLIF: convertir un valor específico a NULL

**Concepto:** `NULLIF(a, b)` retorna `NULL` si `a = b`, de lo contrario retorna `a`. Su uso más común es evitar divisiones por cero: `valor / NULLIF(denominador, 0)` retorna `NULL` en vez de un error cuando el denominador es cero.

In [ ]:
sql("""
SELECT
    Department,
    COUNT(*) AS total,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS rotaron,
    -- NULLIF evita división por cero si algún departamento tuviera 0 empleados
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
        / NULLIF(COUNT(*), 0), 1
    ) AS tasa_rotacion_pct
FROM employee
GROUP BY Department
ORDER BY tasa_rotacion_pct DESC
""")

### Q15 — CAST: convertir tipos de datos

**Concepto:** `CAST(columna AS tipo)` convierte un valor de un tipo a otro. Es fundamental cuando necesitas operar entre columnas de tipos distintos — por ejemplo, dividir un entero entre otro entero en SQLite da entero, no decimal. `CAST(valor AS REAL)` fuerza la división decimal.

In [ ]:
sql("""
SELECT
    Department,
    COUNT(*) AS total,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS rotaron,
    -- Sin CAST: división entera puede dar 0
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) AS tasa_sin_cast,
    -- Con CAST: división decimal correcta
    ROUND(
        CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS REAL)
        / COUNT(*) * 100, 1
    ) AS tasa_con_cast_pct
FROM employee
GROUP BY Department
""")

### Q16 — Empleados con más de 10 años sin promoción

**Concepto:** `WHERE` con múltiples condiciones `AND` — filtra filas individuales antes del agrupamiento. La comparación entre columnas (`YearsSinceLastPromotion >= YearsAtCompany - 1`) detecta casos donde prácticamente no hubo promoción en toda la carrera.

In [ ]:
sql("""
SELECT
    EmployeeID,
    FirstName || ' ' || LastName AS nombre,
    Department,
    JobRole,
    YearsAtCompany,
    YearsSinceLastPromotion,
    Salary,
    Attrition
FROM employee
WHERE YearsAtCompany > 10
  AND YearsSinceLastPromotion >= YearsAtCompany - 1
ORDER BY YearsSinceLastPromotion DESC
LIMIT 20
""")

### Q17 — Distribución porcentual de género por departamento

**Concepto:** Combina `GROUP BY` con dos columnas, `COUNT` condicional y cálculo de porcentaje. `SUM(COUNT(*)) OVER (PARTITION BY Department)` — aunque se adelanta a window functions — es la forma más limpia de calcular el total del grupo para el denominador.

In [ ]:
sql("""
SELECT
    Department,
    Gender,
    COUNT(*) AS total,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY Department), 1
    ) AS pct_en_depto
FROM employee
GROUP BY Department, Gender
ORDER BY Department, Gender
""")

### Q18 — Impacto del overtime en la rotación con múltiples métricas

**Concepto:** Query de cierre del nivel básico — combina varias funciones de agregación (`COUNT`, `SUM`, `AVG`, `ROUND`) con `CASE WHEN` y `GROUP BY` para producir un resumen ejecutivo sobre el impacto de las horas extra.

In [ ]:
sql("""
SELECT
    OverTime,
    COUNT(*) AS total,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS rotaron,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
    ) AS tasa_rotacion_pct,
    ROUND(AVG(Salary), 0)               AS salario_promedio,
    ROUND(AVG(Age), 1)                  AS edad_promedio,
    ROUND(AVG(YearsAtCompany), 1)       AS anos_empresa_promedio
FROM employee
GROUP BY OverTime
ORDER BY tasa_rotacion_pct DESC
""")

---
# 🟡 SECCIÓN 3 — NIVEL INTERMEDIO

Funciones practicadas: `INNER JOIN`, `LEFT JOIN`, `JOIN triple`, subquery escalar, subquery correlacionada, subquery con `IN`, `CTE` simple, `CTE` encadenado, `UNION ALL`, `UNION`, `INTERSECT`, `EXCEPT`, `UPPER`, `LOWER`, `LENGTH`, `TRIM`, `SUBSTR`, `REPLACE`, `strftime`

---

### Bloque 1 — JOINs

### Q01 — INNER JOIN: satisfacción por departamento

**Concepto:** `INNER JOIN` combina filas de dos tablas donde existe coincidencia en la columna de unión. Solo devuelve registros que tienen par en ambas tablas — los empleados sin evaluación quedan excluidos.

In [ ]:
sql("""
SELECT
    e.Department,
    ROUND(AVG(p.JobSatisfaction), 2)          AS avg_job_satisfaction,
    ROUND(AVG(p.EnvironmentSatisfaction), 2)  AS avg_env_satisfaction,
    ROUND(AVG(p.WorkLifeBalance), 2)          AS avg_wlb,
    COUNT(DISTINCT e.EmployeeID)              AS empleados_evaluados
FROM employee e
INNER JOIN performance p ON e.EmployeeID = p.EmployeeID
GROUP BY e.Department
ORDER BY avg_job_satisfaction DESC
""")

### Q02 — LEFT JOIN: detectar empleados sin evaluación

**Concepto:** `LEFT JOIN` devuelve **todos** los registros de la tabla izquierda, incluso si no tienen coincidencia en la derecha — en ese caso los campos de la tabla derecha quedan como `NULL`. Filtrar con `WHERE tabla_derecha.id IS NULL` detecta los registros huérfanos.

In [ ]:
sql("""
SELECT
    e.EmployeeID,
    e.FirstName || ' ' || e.LastName AS nombre,
    e.Department,
    e.HireDate,
    e.Attrition
FROM employee e
LEFT JOIN performance p ON e.EmployeeID = p.EmployeeID
WHERE p.EmployeeID IS NULL
ORDER BY e.HireDate
""")

### Q03 — JOIN con tabla de dimensión: nivel educativo y rotación

**Concepto:** Las tablas de dimensión almacenan catálogos de valores. Hacer JOIN con ellas reemplaza códigos numéricos (`Education = 3`) por etiquetas legibles (`Bachelors`). Es el patrón central de los modelos estrella que usas en Power BI.

In [ ]:
sql("""
SELECT
    e.Department,
    el.EducationLevel,
    COUNT(*) AS total,
    SUM(CASE WHEN e.Attrition = 'Yes' THEN 1 ELSE 0 END) AS rotaron,
    ROUND(
        SUM(CASE WHEN e.Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
    ) AS tasa_pct
FROM employee e
INNER JOIN education_level el ON e.Education = el.EducationLevelID
GROUP BY e.Department, el.EducationLevel
ORDER BY e.Department, tasa_pct DESC
""")

### Q04 — JOIN triple con subquery correlacionada: última evaluación con etiquetas

**Concepto:** JOIN triple — unimos `employee`, `performance` y dos veces `rating_level` con alias distintos (`r` y `r2`). La subquery correlacionada en el `WHERE` garantiza traer solo la evaluación más reciente de cada empleado.

In [ ]:
sql("""
SELECT
    e.EmployeeID,
    e.FirstName || ' ' || e.LastName AS nombre,
    e.Department,
    e.JobRole,
    p.ReviewDate                     AS ultima_evaluacion,
    p.ManagerRating,
    r.RatingLevel                    AS etiqueta_manager,
    p.SelfRating,
    r2.RatingLevel                   AS etiqueta_self
FROM employee e
INNER JOIN performance  p  ON e.EmployeeID    = p.EmployeeID
INNER JOIN rating_level r  ON p.ManagerRating = r.RatingID
INNER JOIN rating_level r2 ON p.SelfRating    = r2.RatingID
WHERE p.ReviewDate = (
    SELECT MAX(p2.ReviewDate)
    FROM performance p2
    WHERE p2.EmployeeID = e.EmployeeID
)
ORDER BY e.Department, p.ManagerRating
LIMIT 20
""")

### Q05 — JOIN con comparación entre columnas: brecha self-rating vs manager

**Concepto:** `WHERE` puede comparar dos columnas de la misma fila. Aquí detectamos empleados donde la autoevaluación supera a la del manager — posible señal de desalineación con el liderazgo.

In [ ]:
sql("""
SELECT
    e.Department,
    e.JobRole,
    COUNT(*) AS casos,
    ROUND(AVG(p.SelfRating - p.ManagerRating), 2) AS brecha_promedio
FROM employee e
INNER JOIN performance p ON e.EmployeeID = p.EmployeeID
WHERE p.SelfRating > p.ManagerRating
GROUP BY e.Department, e.JobRole
ORDER BY casos DESC
""")

### Bloque 2 — Subqueries

### Q06 — Subquery escalar en WHERE: salario sobre el promedio

**Concepto:** Una subquery escalar retorna un único valor que se usa como referencia en el `WHERE` principal. Se evalúa una sola vez para toda la query — más eficiente que calcularla fila a fila.

In [ ]:
sql("""
SELECT
    EmployeeID,
    FirstName || ' ' || LastName AS nombre,
    Department,
    JobRole,
    Salary,
    ROUND(Salary - (SELECT AVG(Salary) FROM employee), 0) AS diferencia_vs_promedio
FROM employee
WHERE Salary > (SELECT AVG(Salary) FROM employee)
ORDER BY Salary DESC
LIMIT 20
""")

### Q07 — Subquery escalar en HAVING: JobRoles sobre promedio general

**Concepto:** La subquery escalar también funciona en el `HAVING` para comparar el resultado de una agregación contra un umbral calculado dinámicamente.

In [ ]:
sql("""
SELECT
    JobRole,
    COUNT(*) AS empleados,
    ROUND(AVG(Salary), 0) AS salario_promedio
FROM employee
GROUP BY JobRole
HAVING AVG(Salary) > (SELECT AVG(Salary) FROM employee)
ORDER BY salario_promedio DESC
""")

### Q08 — Subquery con IN: empleados en departamentos con alta rotación

**Concepto:** `IN (subquery)` filtra filas cuyo valor aparece en el resultado de otra query. Es más flexible que una lista fija porque el conjunto de valores se calcula dinámicamente.

In [ ]:
sql("""
SELECT
    EmployeeID,
    FirstName || ' ' || LastName AS nombre,
    Department,
    JobRole,
    Salary,
    Attrition
FROM employee
WHERE Department IN (
    SELECT Department
    FROM employee
    GROUP BY Department
    HAVING ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1
    ) > 15
)
ORDER BY Department, Attrition DESC
LIMIT 30
""")

### Q09 — EXISTS: empleados con al menos una evaluación con satisfacción muy baja

**Concepto:** `EXISTS` retorna verdadero si la subquery devuelve al menos una fila. Es más eficiente que `IN` con subqueries grandes porque se detiene en el primer resultado encontrado.

In [ ]:
sql("""
SELECT
    e.EmployeeID,
    e.FirstName || ' ' || e.LastName AS nombre,
    e.Department,
    e.JobRole,
    e.Attrition
FROM employee e
WHERE EXISTS (
    SELECT 1
    FROM performance p
    WHERE p.EmployeeID = e.EmployeeID
      AND p.JobSatisfaction = 1
)
ORDER BY e.Department
LIMIT 20
""")

### Bloque 3 — CTEs

### Q10 — CTE simple: perfil del empleado que rota vs el que se queda

**Concepto:** `WITH nombre AS (query)` nombra un resultado intermedio que luego puedes consultar como si fuera una tabla. Hace el código más legible y evita repetir subqueries complejas.

In [ ]:
sql("""
WITH perfil AS (
    SELECT
        Attrition,
        ROUND(AVG(Age), 1)                     AS edad_promedio,
        ROUND(AVG(Salary), 0)                  AS salario_promedio,
        ROUND(AVG(YearsAtCompany), 1)          AS anos_empresa,
        ROUND(AVG(YearsSinceLastPromotion), 1) AS anos_sin_promocion,
        ROUND(AVG(YearsWithCurrManager), 1)    AS anos_con_manager,
        ROUND(AVG(CASE WHEN OverTime = 'Yes' THEN 1.0 ELSE 0 END) * 100, 1) AS pct_overtime
    FROM employee
    GROUP BY Attrition
)
SELECT * FROM perfil
""")

### Q11 — CTE doble: empleados con satisfacción bajo el promedio global

**Concepto:** Puedes encadenar múltiples CTEs separándolos con coma. El segundo CTE puede referenciar al primero. Aquí el primer CTE calcula el promedio global y el segundo lo usa como umbral de filtrado.

In [ ]:
sql("""
WITH avg_global AS (
    SELECT AVG(JobSatisfaction) AS media_global
    FROM performance
),
empleados_bajos AS (
    SELECT
        p.EmployeeID,
        AVG(p.JobSatisfaction) AS avg_satisfaccion
    FROM performance p
    GROUP BY p.EmployeeID
    HAVING AVG(p.JobSatisfaction) < (SELECT media_global FROM avg_global)
)
SELECT
    e.Attrition,
    COUNT(*)                           AS total,
    ROUND(AVG(eb.avg_satisfaccion), 2) AS satisfaccion_promedio
FROM empleados_bajos eb
INNER JOIN employee e ON eb.EmployeeID = e.EmployeeID
GROUP BY e.Attrition
""")

### Q12 — CTE con JOIN: capacitaciones y su efecto en la rotación

**Concepto:** El CTE pre-agrega los datos de performance por empleado. Luego el SELECT principal hace JOIN con employee para cruzar esa información con los datos demográficos y laborales. Divide la lógica en dos pasos claros.

In [ ]:
sql("""
WITH resumen_perf AS (
    SELECT
        EmployeeID,
        ROUND(AVG(JobSatisfaction), 2)       AS avg_satisfaccion,
        SUM(TrainingOpportunitiesTaken)      AS total_capacitaciones
    FROM performance
    GROUP BY EmployeeID
)
SELECT
    CASE
        WHEN rp.total_capacitaciones = 0          THEN 'Sin capacitación'
        WHEN rp.total_capacitaciones BETWEEN 1 AND 5 THEN '1-5 capacitaciones'
        ELSE '6+ capacitaciones'
    END AS nivel_capacitacion,
    COUNT(*) AS empleados,
    ROUND(AVG(rp.avg_satisfaccion), 2) AS satisfaccion_promedio,
    SUM(CASE WHEN e.Attrition = 'Yes' THEN 1 ELSE 0 END) AS rotaron,
    ROUND(
        SUM(CASE WHEN e.Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
    ) AS tasa_rotacion_pct
FROM employee e
INNER JOIN resumen_perf rp ON e.EmployeeID = rp.EmployeeID
GROUP BY nivel_capacitacion
ORDER BY tasa_rotacion_pct DESC
""")

### Q13 — CTE triple encadenado: score simple y clasificación

**Concepto:** Tres CTEs en cadena donde cada uno construye sobre el anterior. El primero agrega performance, el segundo calcula el score, el tercero clasifica. Este patrón es el más cercano a cómo se construyen pipelines de datos en producción.

In [ ]:
sql("""
WITH metricas AS (
    SELECT
        EmployeeID,
        ROUND(AVG(JobSatisfaction), 2)  AS avg_sat,
        ROUND(AVG(WorkLifeBalance), 2)  AS avg_wlb,
        ROUND(AVG(ManagerRating), 2)    AS avg_mgr
    FROM performance
    GROUP BY EmployeeID
),
score AS (
    SELECT
        e.EmployeeID,
        e.FirstName || ' ' || e.LastName AS nombre,
        e.Department,
        e.Attrition,
        (CASE WHEN e.OverTime = 'Yes' THEN 20 ELSE 0 END
         + CASE WHEN m.avg_sat  <= 2 THEN 25 WHEN m.avg_sat  <= 3 THEN 10 ELSE 0 END
         + CASE WHEN m.avg_wlb  <= 2 THEN 20 WHEN m.avg_wlb  <= 3 THEN 8  ELSE 0 END
         + CASE WHEN m.avg_mgr  <= 2 THEN 20 WHEN m.avg_mgr  <= 3 THEN 8  ELSE 0 END
        ) AS score_riesgo
    FROM employee e
    INNER JOIN metricas m ON e.EmployeeID = m.EmployeeID
),
clasificado AS (
    SELECT *,
        CASE
            WHEN score_riesgo >= 60 THEN 'Riesgo Crítico'
            WHEN score_riesgo >= 35 THEN 'Riesgo Alto'
            WHEN score_riesgo >= 15 THEN 'Riesgo Medio'
            ELSE 'Riesgo Bajo'
        END AS categoria_riesgo
    FROM score
)
SELECT
    categoria_riesgo,
    COUNT(*) AS empleados,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS rotaron,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
    ) AS tasa_rotacion_pct
FROM clasificado
GROUP BY categoria_riesgo
ORDER BY MIN(score_riesgo) DESC
""")

### Bloque 4 — Operaciones de conjuntos

### Q14 — UNION ALL: perfil de rotados vs activos en una sola tabla

**Concepto:** `UNION ALL` apila el resultado de dos `SELECT` verticalmente. Ambos deben tener el mismo número de columnas y tipos compatibles. **No elimina duplicados** — es más rápido que `UNION` y es la opción correcta cuando quieres conservar todos los registros de cada segmento.

> Caso de uso típico: consolidar dos segmentos en una sola tabla para exportar a Power BI o hacer una visualización comparativa.

In [ ]:
sql("""
SELECT
    'Rotó'   AS grupo,
    Department,
    JobRole,
    ROUND(AVG(Salary), 0)                  AS salario_promedio,
    ROUND(AVG(YearsAtCompany), 1)          AS anos_empresa,
    ROUND(AVG(YearsSinceLastPromotion), 1) AS anos_sin_promocion
FROM employee
WHERE Attrition = 'Yes'
GROUP BY Department, JobRole

UNION ALL

SELECT
    'Activo' AS grupo,
    Department,
    JobRole,
    ROUND(AVG(Salary), 0),
    ROUND(AVG(YearsAtCompany), 1),
    ROUND(AVG(YearsSinceLastPromotion), 1)
FROM employee
WHERE Attrition = 'No'
GROUP BY Department, JobRole

ORDER BY Department, JobRole, grupo
""")

### Q15 — UNION: lista única de empleados en condición de riesgo

**Concepto:** `UNION` apila dos resultados **eliminando duplicados automáticamente**. Si un empleado cumple ambas condiciones aparece una sola vez. Es más costoso en rendimiento que `UNION ALL` porque debe comparar todas las filas.

> Úsalo cuando necesites una lista única de elementos que cumplen cualquiera de varias condiciones.

In [ ]:
sql("""
SELECT
    e.EmployeeID,
    e.FirstName || ' ' || e.LastName AS nombre,
    e.Department,
    'Overtime + baja satisfacción'   AS condicion_riesgo
FROM employee e
INNER JOIN performance p ON e.EmployeeID = p.EmployeeID
WHERE e.OverTime = 'Yes'
  AND p.JobSatisfaction <= 2

UNION

SELECT
    e.EmployeeID,
    e.FirstName || ' ' || e.LastName,
    e.Department,
    'Sin promoción +5 años'
FROM employee e
WHERE e.YearsSinceLastPromotion >= 5

ORDER BY Department, nombre
""")

### Q16 — INTERSECT: empleados que cumplen AMBAS condiciones simultáneamente

**Concepto:** `INTERSECT` devuelve solo las filas que aparecen en **los dos** resultados. Es más restrictivo que `UNION` — aquí solo verás empleados que hacen overtime con baja satisfacción **y además** llevan 5+ años sin promoción: el perfil de mayor riesgo acumulado.

In [ ]:
sql("""
SELECT e.EmployeeID, e.FirstName || ' ' || e.LastName AS nombre, e.Department
FROM employee e
INNER JOIN performance p ON e.EmployeeID = p.EmployeeID
WHERE e.OverTime = 'Yes'
  AND p.JobSatisfaction <= 2

INTERSECT

SELECT e.EmployeeID, e.FirstName || ' ' || e.LastName, e.Department
FROM employee e
WHERE e.YearsSinceLastPromotion >= 5

ORDER BY Department, nombre
""")

### Q17 — EXCEPT: overtime y baja satisfacción, excluyendo sin promoción

**Concepto:** `EXCEPT` devuelve las filas del **primer conjunto que no aparecen en el segundo**. Útil para aislar un perfil de riesgo específico descartando otro que ya identificaste.

> Resumen de las 4 operaciones de conjuntos:
>
> | Operación | Resultado | Duplicados |
> |---|---|---|
> | `UNION ALL` | Todo de A + todo de B | Los conserva |
> | `UNION` | Todo de A + todo de B | Los elimina |
> | `INTERSECT` | Solo lo que está en A **y** B | Los elimina |
> | `EXCEPT` | Lo que está en A pero **no** en B | Los elimina |

In [ ]:
sql("""
SELECT e.EmployeeID, e.FirstName || ' ' || e.LastName AS nombre, e.Department
FROM employee e
INNER JOIN performance p ON e.EmployeeID = p.EmployeeID
WHERE e.OverTime = 'Yes'
  AND p.JobSatisfaction <= 2

EXCEPT

SELECT e.EmployeeID, e.FirstName || ' ' || e.LastName, e.Department
FROM employee e
WHERE e.YearsSinceLastPromotion >= 5

ORDER BY Department, nombre
""")

### Bloque 5 — Funciones de texto y fecha

### Q18 — Funciones de texto: UPPER, LOWER, LENGTH, TRIM

**Concepto:** SQLite incluye funciones básicas de manipulación de texto. `UPPER` / `LOWER` normalizan mayúsculas, `LENGTH` cuenta caracteres, `TRIM` elimina espacios al inicio y al final — útil para limpiar datos antes de comparaciones o exportaciones.

In [ ]:
sql("""
SELECT
    EmployeeID,
    FirstName,
    LastName,
    UPPER(FirstName)                          AS nombre_mayusculas,
    LOWER(LastName)                           AS apellido_minusculas,
    LENGTH(FirstName || LastName)             AS largo_nombre_completo,
    TRIM('  ' || FirstName || '  ')           AS nombre_sin_espacios,
    UPPER(Department)                         AS depto_mayusculas
FROM employee
LIMIT 10
""")

### Q19 — SUBSTR y REPLACE: extraer y transformar texto

**Concepto:** `SUBSTR(texto, inicio, longitud)` extrae una porción del texto — los índices empiezan en 1. `REPLACE(texto, buscar, reemplazar)` sustituye todas las ocurrencias de una cadena. Ambas son fundamentales para limpiar y estandarizar datos de texto.

In [ ]:
sql("""
SELECT
    EmployeeID,
    -- Extraer los primeros 4 caracteres del ID
    SUBSTR(EmployeeID, 1, 4)                        AS prefijo_id,
    -- Extraer los últimos 4 caracteres del ID
    SUBSTR(EmployeeID, LENGTH(EmployeeID) - 3)      AS sufijo_id,
    JobRole,
    -- Reemplazar espacios por guiones en el JobRole
    REPLACE(JobRole, ' ', '_')                      AS job_role_slug,
    -- Reemplazar 'Manager' por 'Mgr'
    REPLACE(JobRole, 'Manager', 'Mgr')              AS job_role_corto
FROM employee
LIMIT 10
""")

### Q20 — strftime: extraer componentes de fecha y análisis temporal

**Concepto:** `strftime(formato, fecha)` extrae componentes de una fecha en SQLite. Los formatos más usados: `%Y` año, `%m` mes, `%d` día, `%w` día de semana (0=domingo). Permite agrupar contrataciones por año o mes para detectar estacionalidad.

In [ ]:
sql("""
SELECT
    strftime('%Y', HireDate)        AS anio_contratacion,
    strftime('%m', HireDate)        AS mes_contratacion,
    CASE strftime('%w', HireDate)
        WHEN '0' THEN 'Domingo'
        WHEN '1' THEN 'Lunes'
        WHEN '2' THEN 'Martes'
        WHEN '3' THEN 'Miércoles'
        WHEN '4' THEN 'Jueves'
        WHEN '5' THEN 'Viernes'
        WHEN '6' THEN 'Sábado'
    END AS dia_semana_contratacion,
    COUNT(*) AS contratados,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS rotaron
FROM employee
GROUP BY anio_contratacion, mes_contratacion
ORDER BY anio_contratacion, mes_contratacion
""")

---
# 🔴 SECCIÓN 4 — NIVEL AVANZADO

Funciones practicadas: `RANK()`, `DENSE_RANK()`, `ROW_NUMBER()`, `NTILE()`, `LAG()`, `LEAD()`, `FIRST_VALUE()`, `LAST_VALUE()`, `AVG/SUM/COUNT/MIN/MAX OVER`, `ROWS BETWEEN`, `PERCENT_RANK()`, `CUME_DIST()`, acumulados, Pareto, cohortes

> **Nota:** `ROLLUP` no está soportado en SQLite. En PostgreSQL o SQL Server se usa `GROUP BY ROLLUP(col1, col2)` para generar subtotales automáticos por nivel de agrupamiento.

---

### Bloque 1 — Funciones de ranking

### Q01 — RANK, DENSE_RANK y ROW_NUMBER

**Concepto:** Las tres funciones de ranking aplicadas al mismo conjunto:
- `RANK()`: asigna el mismo número a empates y **salta** el siguiente (1, 1, 3)
- `DENSE_RANK()`: asigna el mismo número a empates pero **no salta** (1, 1, 2)
- `ROW_NUMBER()`: número único sin importar empates (1, 2, 3)

`PARTITION BY` reinicia el contador en cada grupo.

In [ ]:
sql("""
SELECT
    EmployeeID,
    FirstName || ' ' || LastName AS nombre,
    Department,
    JobRole,
    Salary,
    RANK()       OVER (PARTITION BY Department ORDER BY Salary DESC) AS rank_salario,
    DENSE_RANK() OVER (PARTITION BY Department ORDER BY Salary DESC) AS dense_rank_salario,
    ROW_NUMBER() OVER (PARTITION BY Department ORDER BY Salary DESC) AS row_num
FROM employee
ORDER BY Department, rank_salario
LIMIT 30
""")

### Q02 — NTILE: dividir empleados en cuartiles de salario

**Concepto:** `NTILE(n)` divide cada partición en `n` grupos de tamaño igual (o lo más igual posible). `NTILE(4)` crea cuartiles, `NTILE(10)` deciles. El grupo 1 corresponde al extremo superior del orden definido.

In [ ]:
sql("""
SELECT
    cuartil,
    COUNT(*) AS empleados,
    MIN(Salary) AS salario_min,
    MAX(Salary) AS salario_max,
    ROUND(AVG(Salary), 0) AS salario_promedio,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS rotaron,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
    ) AS tasa_rotacion_pct
FROM (
    SELECT
        Salary, Attrition,
        NTILE(4) OVER (ORDER BY Salary DESC) AS cuartil
    FROM employee
)
GROUP BY cuartil
ORDER BY cuartil
""")

### Q03 — AVG OVER PARTITION BY: salario vs promedio del departamento

**Concepto:** `AVG() OVER (PARTITION BY col)` calcula el promedio del grupo para cada fila sin colapsar el resultado. A diferencia del `GROUP BY` que reduce N filas a 1, la window function mantiene todas las filas y agrega el promedio como columna adicional.

In [ ]:
sql("""
SELECT
    EmployeeID,
    FirstName || ' ' || LastName AS nombre,
    Department,
    Salary,
    ROUND(AVG(Salary) OVER (PARTITION BY Department), 0) AS avg_depto,
    ROUND(MIN(Salary) OVER (PARTITION BY Department), 0) AS min_depto,
    ROUND(MAX(Salary) OVER (PARTITION BY Department), 0) AS max_depto,
    COUNT(*)      OVER (PARTITION BY Department)         AS total_depto,
    Salary - ROUND(AVG(Salary) OVER (PARTITION BY Department), 0) AS diferencia,
    ROUND(
        (Salary - AVG(Salary) OVER (PARTITION BY Department)) * 100.0
        / AVG(Salary) OVER (PARTITION BY Department), 1
    ) AS desviacion_pct
FROM employee
ORDER BY Department, desviacion_pct DESC
LIMIT 30
""")

### Bloque 2 — Funciones de desplazamiento

### Q04 — LAG y LEAD: evolución de satisfacción entre evaluaciones

**Concepto:**
- `LAG(col, n)`: trae el valor `n` filas **antes** dentro de la partición
- `LEAD(col, n)`: trae el valor `n` filas **después**

Permiten calcular cambios periodo a periodo sin hacer self-joins. El primer registro de cada empleado tendrá `NULL` en `LAG` porque no hay fila anterior.

In [ ]:
sql("""
SELECT
    EmployeeID,
    ReviewDate,
    JobSatisfaction,
    LAG(JobSatisfaction)  OVER (PARTITION BY EmployeeID ORDER BY ReviewDate) AS sat_anterior,
    LEAD(JobSatisfaction) OVER (PARTITION BY EmployeeID ORDER BY ReviewDate) AS sat_siguiente,
    JobSatisfaction
        - LAG(JobSatisfaction) OVER (PARTITION BY EmployeeID ORDER BY ReviewDate) AS cambio,
    CASE
        WHEN JobSatisfaction > LAG(JobSatisfaction) OVER (PARTITION BY EmployeeID ORDER BY ReviewDate)
             THEN 'Mejora'
        WHEN JobSatisfaction < LAG(JobSatisfaction) OVER (PARTITION BY EmployeeID ORDER BY ReviewDate)
             THEN 'Baja'
        ELSE 'Sin cambio'
    END AS tendencia
FROM performance
ORDER BY EmployeeID, ReviewDate
LIMIT 30
""")

### Q05 — FIRST_VALUE y LAST_VALUE: primera y última evaluación por empleado

**Concepto:** `FIRST_VALUE()` y `LAST_VALUE()` traen el primer y último valor de la ventana. Para que `LAST_VALUE` funcione correctamente debes definir `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` — de lo contrario la ventana por defecto solo llega hasta la fila actual.

In [ ]:
sql("""
SELECT DISTINCT
    e.EmployeeID,
    e.FirstName || ' ' || e.LastName AS nombre,
    e.Attrition,
    FIRST_VALUE(p.JobSatisfaction) OVER (
        PARTITION BY p.EmployeeID ORDER BY p.ReviewDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS satisfaccion_inicial,
    LAST_VALUE(p.JobSatisfaction) OVER (
        PARTITION BY p.EmployeeID ORDER BY p.ReviewDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS satisfaccion_final,
    LAST_VALUE(p.JobSatisfaction) OVER (
        PARTITION BY p.EmployeeID ORDER BY p.ReviewDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) - FIRST_VALUE(p.JobSatisfaction) OVER (
        PARTITION BY p.EmployeeID ORDER BY p.ReviewDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS variacion_total
FROM employee e
INNER JOIN performance p ON e.EmployeeID = p.EmployeeID
ORDER BY variacion_total
LIMIT 20
""")

### Q06 — LAG con offset: comparar evaluación actual vs la de hace 2 períodos

**Concepto:** `LAG(col, 2)` salta dos filas hacia atrás en lugar de una. El tercer parámetro opcional `LAG(col, n, valor_default)` reemplaza el `NULL` de los primeros registros con un valor por defecto — útil para evitar nulos en cálculos posteriores.

In [ ]:
sql("""
SELECT
    EmployeeID,
    ReviewDate,
    JobSatisfaction                                                        AS sat_actual,
    LAG(JobSatisfaction, 1, JobSatisfaction) OVER
        (PARTITION BY EmployeeID ORDER BY ReviewDate)                      AS sat_periodo_anterior,
    LAG(JobSatisfaction, 2, JobSatisfaction) OVER
        (PARTITION BY EmployeeID ORDER BY ReviewDate)                      AS sat_hace_2_periodos,
    JobSatisfaction - LAG(JobSatisfaction, 2, JobSatisfaction) OVER
        (PARTITION BY EmployeeID ORDER BY ReviewDate)                      AS cambio_2_periodos
FROM performance
ORDER BY EmployeeID, ReviewDate
LIMIT 30
""")

### Bloque 3 — Ventanas con ROWS BETWEEN

### Q07 — Promedio móvil de satisfacción (ventana de 3 períodos)

**Concepto:** `ROWS BETWEEN N PRECEDING AND CURRENT ROW` define una ventana deslizante que incluye las N filas anteriores más la actual. El promedio móvil suaviza fluctuaciones y revela tendencias. `UNBOUNDED PRECEDING` expande la ventana desde el inicio de la partición — genera el promedio histórico acumulado.

In [ ]:
sql("""
SELECT
    EmployeeID,
    ReviewDate,
    JobSatisfaction,
    ROUND(AVG(JobSatisfaction) OVER (
        PARTITION BY EmployeeID
        ORDER BY ReviewDate
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS avg_movil_3,
    ROUND(AVG(JobSatisfaction) OVER (
        PARTITION BY EmployeeID
        ORDER BY ReviewDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS avg_acumulado,
    SUM(TrainingOpportunitiesTaken) OVER (
        PARTITION BY EmployeeID
        ORDER BY ReviewDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS capacitaciones_acumuladas
FROM performance
ORDER BY EmployeeID, ReviewDate
LIMIT 30
""")

### Q08 — MIN y MAX en ventana: rango histórico de satisfacción por empleado

**Concepto:** `MIN()` y `MAX()` también funcionan como window functions. Con `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` calculan el mínimo y máximo histórico hasta la fila actual — útil para detectar si el empleado está en su peor o mejor momento.

In [ ]:
sql("""
SELECT
    EmployeeID,
    ReviewDate,
    JobSatisfaction,
    MIN(JobSatisfaction) OVER (
        PARTITION BY EmployeeID
        ORDER BY ReviewDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS min_historico,
    MAX(JobSatisfaction) OVER (
        PARTITION BY EmployeeID
        ORDER BY ReviewDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS max_historico,
    CASE
        WHEN JobSatisfaction = MIN(JobSatisfaction) OVER (
            PARTITION BY EmployeeID ORDER BY ReviewDate
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
        THEN 'Mínimo histórico'
        WHEN JobSatisfaction = MAX(JobSatisfaction) OVER (
            PARTITION BY EmployeeID ORDER BY ReviewDate
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
        THEN 'Máximo histórico'
        ELSE ''
    END AS nota
FROM performance
ORDER BY EmployeeID, ReviewDate
LIMIT 30
""")

### Q09 — COUNT OVER: número de evaluaciones acumuladas por empleado

**Concepto:** `COUNT() OVER` con ventana acumulada cuenta cuántas filas han aparecido hasta la fila actual dentro de la partición. Permite ver la evolución del número de evaluaciones a lo largo del tiempo.

In [ ]:
sql("""
SELECT
    EmployeeID,
    ReviewDate,
    JobSatisfaction,
    COUNT(*) OVER (
        PARTITION BY EmployeeID
        ORDER BY ReviewDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS num_evaluacion_acumulada,
    COUNT(*) OVER (PARTITION BY EmployeeID) AS total_evaluaciones_empleado,
    COUNT(*) OVER ()                        AS total_evaluaciones_global
FROM performance
ORDER BY EmployeeID, ReviewDate
LIMIT 30
""")

### Bloque 4 — Distribución estadística

### Q10 — PERCENT_RANK: posición percentil del salario

**Concepto:** `PERCENT_RANK()` calcula la posición relativa de cada fila como un valor entre 0 y 1. La fórmula es `(rank - 1) / (total_filas - 1)`. Un valor de 0.9 significa que el empleado tiene un salario mayor al 90% de sus compañeros del departamento.

In [ ]:
sql("""
SELECT
    EmployeeID,
    FirstName || ' ' || LastName AS nombre,
    Department,
    Salary,
    ROUND(PERCENT_RANK() OVER (PARTITION BY Department ORDER BY Salary) * 100, 1) AS percentil_en_depto,
    ROUND(PERCENT_RANK() OVER (ORDER BY Salary) * 100, 1)                         AS percentil_global
FROM employee
ORDER BY Department, Salary DESC
LIMIT 30
""")

### Q11 — CUME_DIST: distribución acumulada de salarios

**Concepto:** `CUME_DIST()` calcula qué proporción de filas tiene un valor menor o igual al actual. A diferencia de `PERCENT_RANK`, incluye la fila actual en el numerador, por lo que el último valor siempre es 1.0. Útil para construir curvas de distribución acumulada.

In [ ]:
sql("""
SELECT
    EmployeeID,
    FirstName || ' ' || LastName AS nombre,
    Department,
    Salary,
    ROUND(CUME_DIST() OVER (PARTITION BY Department ORDER BY Salary) * 100, 1) AS cume_dist_depto_pct,
    ROUND(CUME_DIST() OVER (ORDER BY Salary) * 100, 1)                         AS cume_dist_global_pct
FROM employee
ORDER BY Department, Salary
LIMIT 30
""")

### Q12 — Comparación PERCENT_RANK vs CUME_DIST vs NTILE

**Concepto:** Las tres funciones de distribución en una sola query para comparar sus diferencias:
- `PERCENT_RANK`: posición relativa, el primero es siempre 0
- `CUME_DIST`: proporción acumulada, el último es siempre 1
- `NTILE(100)`: percentil discreto — divide en 100 grupos iguales

In [ ]:
sql("""
SELECT
    Salary,
    Department,
    ROUND(PERCENT_RANK() OVER (ORDER BY Salary) * 100, 1) AS percent_rank_pct,
    ROUND(CUME_DIST()    OVER (ORDER BY Salary) * 100, 1) AS cume_dist_pct,
    NTILE(100)           OVER (ORDER BY Salary)           AS ntile_percentil
FROM employee
ORDER BY Salary
LIMIT 20
""")

### Bloque 5 — Pareto y acumulados

### Q13 — Pareto de headcount por JobRole

**Concepto:** El análisis de Pareto identifica qué pocos grupos concentran la mayoría del volumen (regla 80/20). Se construye con `SUM() OVER (ORDER BY ... ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` para el acumulado, dividido entre el total global.

In [ ]:
sql("""
WITH conteo AS (
    SELECT JobRole, COUNT(*) AS total
    FROM employee
    GROUP BY JobRole
),
pareto AS (
    SELECT
        JobRole,
        total,
        SUM(total) OVER () AS gran_total,
        SUM(total) OVER (
            ORDER BY total DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS acumulado,
        ROUND(total * 100.0 / SUM(total) OVER (), 1) AS pct_individual,
        ROUND(
            SUM(total) OVER (
                ORDER BY total DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) * 100.0 / SUM(total) OVER (), 1
        ) AS pct_acumulado
    FROM conteo
)
SELECT
    JobRole, total, pct_individual, acumulado, pct_acumulado,
    CASE WHEN pct_acumulado <= 80 THEN 'Top 80%' ELSE 'Cola 20%' END AS segmento_pareto
FROM pareto
ORDER BY total DESC
""")

### Q14 — Pareto de rotación: JobRoles que concentran el 80% de las bajas

**Concepto:** Mismo patrón de Pareto pero aplicado a las bajas por rotación. Identifica en qué roles se concentra el problema de attrition — información crítica para diseñar intervenciones de retención focalizadas.

In [ ]:
sql("""
WITH rotacion AS (
    SELECT
        JobRole,
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS bajas
    FROM employee
    GROUP BY JobRole
),
pareto_rotacion AS (
    SELECT
        JobRole, bajas,
        SUM(bajas) OVER () AS total_bajas,
        SUM(bajas) OVER (
            ORDER BY bajas DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS acum_bajas,
        ROUND(bajas * 100.0 / SUM(bajas) OVER (), 1) AS pct_individual,
        ROUND(
            SUM(bajas) OVER (
                ORDER BY bajas DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) * 100.0 / SUM(bajas) OVER (), 1
        ) AS pct_acumulado
    FROM rotacion
)
SELECT
    JobRole, bajas, pct_individual, pct_acumulado,
    CASE WHEN pct_acumulado <= 80 THEN 'Crítico (80%)' ELSE 'Resto' END AS clasificacion
FROM pareto_rotacion
ORDER BY bajas DESC
""")

### Q15 — Participación % de salario por departamento con acumulado

**Concepto:** `SUM() OVER ()` sin partición calcula el total global — útil para calcular la participación porcentual de cada grupo sobre el total. Combinado con el acumulado permite ver qué departamentos concentran la mayor parte de la masa salarial.

In [ ]:
sql("""
WITH masa AS (
    SELECT
        Department,
        SUM(Salary) AS masa_salarial,
        COUNT(*)    AS empleados
    FROM employee
    GROUP BY Department
)
SELECT
    Department,
    empleados,
    masa_salarial,
    ROUND(masa_salarial * 100.0 / SUM(masa_salarial) OVER (), 1) AS pct_masa_salarial,
    SUM(masa_salarial) OVER (
        ORDER BY masa_salarial DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS acumulado_masa,
    ROUND(
        SUM(masa_salarial) OVER (
            ORDER BY masa_salarial DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) * 100.0 / SUM(masa_salarial) OVER (), 1
    ) AS pct_acumulado
FROM masa
ORDER BY masa_salarial DESC
""")

### Bloque 6 — Análisis compuesto

### Q16 — Análisis de cohorte por año de contratación

**Concepto:** Una cohorte agrupa personas que comparten un evento en el mismo período. Aquí agrupamos por año de contratación y analizamos rotación, salario y antigüedad de cada cohorte. El acumulado de contratados muestra el crecimiento histórico de la empresa.

In [ ]:
sql("""
WITH cohorte AS (
    SELECT
        strftime('%Y', HireDate)                                      AS anio,
        COUNT(*)                                                       AS contratados,
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)            AS rotaron,
        ROUND(AVG(Salary), 0)                                          AS salario_promedio,
        ROUND(AVG(YearsAtCompany), 1)                                  AS anos_promedio
    FROM employee
    GROUP BY anio
)
SELECT
    anio,
    contratados,
    rotaron,
    ROUND(rotaron * 100.0 / contratados, 1)    AS tasa_rotacion_pct,
    salario_promedio,
    anos_promedio,
    SUM(contratados) OVER (
        ORDER BY anio
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS contratados_acum,
    ROUND(
        SUM(contratados) OVER (
            ORDER BY anio
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) * 100.0 / SUM(contratados) OVER (), 1
    ) AS pct_acum_headcount
FROM cohorte
ORDER BY anio
""")

### Q17 — Top 10% salarial con rating de manager bajo por departamento

**Concepto:** `NTILE(10)` aplicado dentro de cada departamento crea deciles salariales. El decil 1 es el 10% con salario más alto. Combinado con CTEs encadenados y JOIN con rating_level detecta un perfil de riesgo específico: empleados bien pagados pero con evaluación de liderazgo deficiente.

In [ ]:
sql("""
WITH salario_ntile AS (
    SELECT
        EmployeeID, Department, Salary,
        NTILE(10) OVER (PARTITION BY Department ORDER BY Salary DESC) AS decil
    FROM employee
),
ultima_eval AS (
    SELECT EmployeeID, ManagerRating, ReviewDate
    FROM performance p
    WHERE ReviewDate = (
        SELECT MAX(p2.ReviewDate) FROM performance p2
        WHERE p2.EmployeeID = p.EmployeeID
    )
)
SELECT
    e.EmployeeID,
    e.FirstName || ' ' || e.LastName AS nombre,
    e.Department,
    e.JobRole,
    e.Salary,
    s.decil,
    ue.ManagerRating,
    r.RatingLevel,
    e.Attrition
FROM employee e
INNER JOIN salario_ntile s ON e.EmployeeID = s.EmployeeID
INNER JOIN ultima_eval  ue ON e.EmployeeID = ue.EmployeeID
INNER JOIN rating_level  r ON ue.ManagerRating = r.RatingID
WHERE s.decil = 1
  AND ue.ManagerRating <= 2
ORDER BY e.Department, e.Salary DESC
""")

### Q18 — Score de riesgo de rotación con ranking completo

**Concepto:** Query de síntesis final que combina múltiples señales en un índice compuesto ponderado. Cada factor de riesgo suma puntos al score. El resultado se rankea globalmente y por departamento con `RANK()`, y se segmenta en cuartiles con `NTILE(4)`.

> Este tipo de score es el que usarías como input para un modelo de ML o como KPI en un dashboard de RRHH para identificar empleados en riesgo antes de que renuncien.

In [ ]:
sql("""
WITH avg_perf AS (
    SELECT
        EmployeeID,
        ROUND(AVG(JobSatisfaction), 2)  AS avg_job_sat,
        ROUND(AVG(WorkLifeBalance), 2)  AS avg_wlb,
        ROUND(AVG(ManagerRating), 2)    AS avg_mgr_rating
    FROM performance
    GROUP BY EmployeeID
),
score AS (
    SELECT
        e.EmployeeID,
        e.FirstName || ' ' || e.LastName AS nombre,
        e.Department,
        e.JobRole,
        e.Attrition,
        ap.avg_job_sat,
        ap.avg_wlb,
        ap.avg_mgr_rating,
        ROUND(
            (CASE WHEN e.OverTime = 'Yes'             THEN 20 ELSE 0 END)
          + (CASE WHEN e.YearsSinceLastPromotion >= 5 THEN 15 ELSE 0 END)
          + (CASE WHEN ap.avg_job_sat <= 2            THEN 25
                  WHEN ap.avg_job_sat <= 3            THEN 10 ELSE 0 END)
          + (CASE WHEN ap.avg_wlb <= 2               THEN 20
                  WHEN ap.avg_wlb <= 3               THEN 8  ELSE 0 END)
          + (CASE WHEN ap.avg_mgr_rating <= 2         THEN 20
                  WHEN ap.avg_mgr_rating <= 3         THEN 8  ELSE 0 END)
        , 0) AS score_riesgo
    FROM employee e
    INNER JOIN avg_perf ap ON e.EmployeeID = ap.EmployeeID
)
SELECT
    EmployeeID, nombre, Department, JobRole, Attrition,
    score_riesgo,
    RANK()   OVER (ORDER BY score_riesgo DESC)                         AS ranking_global,
    RANK()   OVER (PARTITION BY Department ORDER BY score_riesgo DESC) AS ranking_depto,
    NTILE(4) OVER (ORDER BY score_riesgo DESC)                         AS cuartil_riesgo
FROM score
ORDER BY score_riesgo DESC
LIMIT 30
""")

---

## ✅ ¡Completaste las 56 queries!

### Resumen de lo que practicaste

| Nivel | Queries | Funciones principales |
|---|---|---|
| 🟢 Básico | 18 | `SELECT`, `DISTINCT`, `LIKE`, `IN`, `BETWEEN`, `NOT IN`, `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`, `ROUND`, `CASE WHEN`, `GROUP BY`, `HAVING`, `COALESCE`, `NULLIF`, `CAST` |
| 🟡 Intermedio | 20 | `INNER JOIN`, `LEFT JOIN`, `JOIN triple`, subquery escalar, subquery correlacionada, `IN (subquery)`, `EXISTS`, `CTE` simple/doble/triple, `UNION ALL`, `UNION`, `INTERSECT`, `EXCEPT`, `UPPER`, `LOWER`, `LENGTH`, `TRIM`, `SUBSTR`, `REPLACE`, `strftime` |
| 🔴 Avanzado | 18 | `RANK`, `DENSE_RANK`, `ROW_NUMBER`, `NTILE`, `LAG`, `LEAD`, `FIRST_VALUE`, `LAST_VALUE`, `AVG/SUM/COUNT/MIN/MAX OVER`, `ROWS BETWEEN`, `PERCENT_RANK`, `CUME_DIST`, Pareto, cohortes, score compuesto |

### Próximos pasos sugeridos
- Modifica los umbrales del `HAVING` o los pesos del score de riesgo y observa cómo cambian los resultados
- Exporta cualquier resultado a CSV: `sql('...').to_csv('resultado.csv', index=False)`
- Conecta los resultados a Power BI o Tableau para visualizarlos
- Prueba las mismas queries en PostgreSQL o SQL Server y nota las diferencias de sintaxis

---
*Repositorio: [github.com/rafael-milanes/sql-hr-analytics](https://github.com/rafael-milanes/sql-hr-analytics)*